Парсер новостей с сайта

In [1]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import quote
import cloudscraper
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import chardet
import re

In [2]:
def click_load_more(driver):
    """Ищет и нажимает кнопку 'Загрузить еще'"""
    load_more_selectors = [
        "button.load-more",
        "button[class*='load-more']",
        "button:contains('Загрузить еще')",
        "button:contains('Load more')",
        "a.load-more",
        ".load-more-btn",
        "button.js-load-more",
        "app-button-load-more"
    ]
    
    for selector in load_more_selectors:
        try:
            load_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, selector))
            )
            driver.execute_script("arguments[0].click();", load_button)
            print(f"Нажата кнопка: {selector}")
            return True
        except:
            continue
    
    return False

In [3]:
# pip install selenium webdriver-manager beautifulsoup4
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    TimeoutException, ElementClickInterceptedException,
    StaleElementReferenceException, NoSuchElementException
)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

In [4]:
def open_page(url, driver):
    
    # print(url)
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36', 
               'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'ru-RU,ru;q=0.8,en-US;q=0.5,en;q=0.3',
        'Accept-Encoding': 'gzip, deflate, br',}
    response = requests.get(url, headers=headers)
    encoding = chardet.detect(response.content)['encoding']
    response.encoding = encoding if encoding else 'utf-8'
    if "www.fontanka.ru" in url:
        soup=BeautifulSoup(response.text, 'html.parser')
        return soup

    driver.set_page_load_timeout(300) 
    driver.implicitly_wait(20)
    driver.set_script_timeout(60)

    retries=3
    for attempt in range(retries):
        try:
            driver.get(url)
            time.sleep(5)
            WebDriverWait(driver, 180).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            # driver.quit()
            return soup
        except TimeoutException:
            print(f"Таймаут при загрузке {url}, попытка {attempt + 1}")
            time.sleep(10)  # подождать перед повтором
    raise Exception(f"Не удалось загрузить {url} после {retries} попыток")

    # html = driver.page_source
    # soup = BeautifulSoup(html, 'html.parser')
    # driver.quit()
    # return soup

In [5]:
def parse_fontanka(driver, company, date_from, date_to, name=""):
    if name=="":
        name=company

    print(type(name), name)
    company_name = name.replace(" ", "+")
    url = f"https://www.fontanka.ru/cgi-bin/search.scgi?query={company_name}&fdate={date_from}&tdate={date_to}&sortt=date"
    print(url)
    soup=open_page(url, driver)
    # headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    # response = requests.get(url, headers=headers)
    # soup = BeautifulSoup(response.text, "html.parser")



    articles = []
    # print(soup)
    print(soup.find_all('li', class_="K5alj MPap"))
    for item in soup.find_all('li', class_='K5alj MPap'):
        title_elem = item.find('div', class_='K5alx')
        title_elem = title_elem.find('a')
        title = title_elem.text.strip() if title_elem else ''
        # print(title_elem['href'])
        if 'https://www.fontanka.ru' in title_elem['href'] or 'https://' in title_elem['href']:
            link=title_elem['href']
        else:
            link = 'https://www.fontanka.ru' + title_elem['href'] if title_elem else ''
        data_time = item.find('time')
        date_str = data_time.find('span').text.strip() if data_time.find('span') else ''  # Формат: DD.MM.YYYY
        # summary = item.find('p', class_='snippet').text.strip() if item.find('p', class_='snippet') else ''
        
        # Фильтр по дате
        try:
            article_date = datetime.strptime(date_str, '%d.%m.%Y')
            if not (datetime.strptime(date_from, '%Y-%m-%d') <= article_date <= datetime.strptime(date_to, '%Y-%m-%d')):
                continue
        except ValueError:
            pass
        
        print(title)
        print(date_str)
        print(link)


        summary=""

        # response = requests.get(link, headers=headers)
        # soup = BeautifulSoup(response.text, "html.parser")
        # print(soup)

        soup=open_page(link, driver)
        article_text=soup.find_all('div', class_="content_gSlSK")
        # print("Article", article_text)
        for item in article_text:
            texts = item.find_all('p')
            for t in texts:
                text=t.text.strip()
                summary=summary+"\n"+text

        # print(summary)

        articles.append({
            'NewsFinder': "парсер",
            'CompanyName': company,
            'NewsTitle': title,
            'NewsDate': date_str,
            'NewsSource': 'fontanka.ru',
            'NewsURL': link,
            'NewsText': summary,
            "CompanyVar":name
        })


    j=2
    while True:
        try:
            url = f"https://www.fontanka.ru/cgi-bin/search.scgi?query={company_name}&fdate={date_from}&tdate={date_to}&sortt=date&offset={j}"
            print(url)
            soup=open_page(url, driver)
            print(soup.find_all('li', class_='K5alj MPap'))
            if not(soup.find_all('li', class_='K5alj MPap')): break
            for item in soup.find_all('li', class_='K5alj MPap'):
                # print(item)
                title_elem = item.find('div', class_='K5alx')
                title_elem = title_elem.find('a')
                title = title_elem.text.strip() if title_elem else ''
                if 'https://www.fontanka.ru' in title_elem['href'] or 'https://' in title_elem['href']:
                    link=title_elem['href']
                else:
                    link = 'https://www.fontanka.ru' + title_elem['href'] if title_elem else ''
                data_time = item.find('time')
                date_str = data_time.find('span').text.strip() if data_time.find('span') else ''  # Формат: DD.MM.YYYY
                # summary = item.find('p', class_='snippet').text.strip() if item.find('p', class_='snippet') else ''
                
                # Фильтр по дате
                try:
                    article_date = datetime.strptime(date_str, '%d.%m.%Y')
                    if not (datetime.strptime(date_from, '%Y-%m-%d') <= article_date <= datetime.strptime(date_to, '%Y-%m-%d')):
                        continue
                except ValueError:
                    pass
                
                print(title)
                print(date_str)
                print(link)


                summary=""

                # response = requests.get(link, headers=headers)
                # soup = BeautifulSoup(response.text, "html.parser")
                # print(soup)

                soup=open_page(link, driver)
                article_text=soup.find_all('div', class_="content_gSlSK")
                # print("Article", article_text)
                for item in article_text:
                    texts = item.find_all('p')
                    for t in texts:
                        text=t.text.strip()
                        summary=summary+"\n"+text

                print(summary)

                articles.append({
                    'NewsFinder': "парсер",
                    'CompanyName': company,
                    'NewsTitle': title,
                    'NewsDate': date_str,
                    'NewsSource': 'fontanka.ru',
                    'NewsURL': link,
                    'NewsText': summary,
                    "CompanyVar":name
                })
            j+=1
        except: 
            print("break")
            break
            


    return articles

In [6]:
def parse_dp(driver, company, date_from, date_to, name=""):
    if name=="":
        name=company
    

    date_from_q = datetime.strptime(date_from, "%Y-%m-%d")
    date_to_q = datetime.strptime(date_to, "%Y-%m-%d")
    date_from_q = quote(date_from_q.strftime("%d/%m/%Y"))
    date_to_q = quote(date_to_q.strftime("%d/%m/%Y"))
    company_name = name.replace(" ", "%20")

    url = f"https://www.dp.ru/search?dateEnd={date_to_q}&dateStart={date_from_q}&searchString={company_name}&isTag=false&isAuthor=false&sortType=2&take=200"
    print(url)
    soup=open_page(url, driver)
    articles = []
    print("Найдено новостей:", len(soup.find_all('app-article-line', attrs={'_ngcontent-dpruapp-c58': '', '_nghost-dpruapp-c57': ''})))
    for item in soup.find_all('app-article-line', attrs={'_ngcontent-dpruapp-c58': '', '_nghost-dpruapp-c57': ''}):
        title_elem = item.find('div')
        title_elem = title_elem.find('a', class_="article-fav-block-headline", attrs={'_ngcontent-dpruapp-c57': ''})
        title = title_elem.find('span').text.strip() if title_elem else ''
        link = 'https://www.dp.ru' + title_elem['href'] if title_elem else ''

        date=item.find('div').find('div', class_="article-fav-block-publication-date text-nowrap", attrs={'_ngcontent-dpruapp-c57': ''})
        date_str = date.find('span').text.strip() if date.find('span') else ''  # Формат: DD.MM.YYYY
        summary = item.find('p', class_='summary').text.strip() if item.find('p', class_='summary') else ''
        
        # Фильтр по дате
        try:
            article_date = datetime.strptime(date_str, '%d.%m.%Y')
            if not (datetime.strptime(date_from, '%Y-%m-%d') <= article_date <= datetime.strptime(date_to, '%Y-%m-%d')):
                continue
        except ValueError:
            pass
        
        print(title)
        print(date_str)
        print(link)

        summary=""
        soup=open_page(link, driver)
        article_text=soup.find_all('div', class_="paragraph-wrapper", attrs={'_ngcontent-dpruapp-c95':""})
        for item in article_text:
            texts = item.find_all('div', class_="paragraph paragraph-text")
            for t in texts:
                text=t.text.strip()
                summary=summary+"\n"+text


        articles.append({
            'NewsFinder': "парсер",
            'CompanyName': company,
            'NewsTitle': title,
            'NewsDate': date_str,
            'NewsSource': 'dp.ru',
            'NewsURL': link,
            'NewsText': summary,
            "CompanyVar":name
        })
    return articles

In [7]:
import re
def extra_names(name):
    result=[]
    if ',' in name:
        v1 = name.split(',')[0]
        result.append(v1)

        if "-" in v1:
            v3 = v1.replace('-', ' ')
            if v3 not in result:
                result.append(v3)

    match = re.search(r'\((.*?)\)', name)
    if match:
        v2 = match.group(1)
        if v2 not in result:
            result.append(v2)

    return result


print(extra_names("ПАО (какао)"))

['какао']


In [11]:
date_from = "2025-01-01"
date_to = "2025-12-31"
industry="Комунальные услуги"
name="Companies - Коммунальные услуги - 78_47_83"

file_names=pd.read_csv(name+".csv")
# print(file_names)

# names=file_names.iloc[, 0:1]
names = file_names[lambda df: df.iloc[:, 3].isin([78, 47])].iloc[7:, :]
# names=names[7:, :]
print(names, type(names))

options = webdriver.ChromeOptions()
options.add_argument("--headless")  # без окна
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-blink-features=AutomationControlled")

options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins")
options.add_argument("--disable-background-timer-throttling")
options.add_argument("--disable-renderer-backgrounding")
options.add_argument("--disable-backgrounding-occluded-windows")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# for name in names:
for i in range(len(names)):
    name=names.iloc[i, 0]
    print("name", name)
    # extra_name=extra_names(name)
    extra_name=names.iloc[i, -1].split(",")
    print("extra names", extra_name)
    inn=names.iloc[i, 1]
    print('inn', inn)

    news_fontanka = parse_fontanka(driver, name, date_from, date_to)
    
    
    extra_fontanka=[]
    for i in extra_name:
        extra=parse_fontanka(driver, name, date_from, date_to, i)
        extra_fontanka=extra_fontanka+extra
    duble=[]
    for j in range(len(extra_fontanka)):
        if extra_fontanka[j] in news_fontanka:
                    duble.append(j)
    for k in sorted(duble, reverse=True):
        del extra_fontanka[k]

    news_fontanka_extra=[]
    try:
        if not(pd.isna(names.iloc[i, 1])):
            news_fontanka_extra = parse_fontanka(driver, name, date_from, date_to, names.iloc[i, 1])
            duble=[]
            for j in range(len(news_fontanka_extra)):
                if news_fontanka_extra[j] in news_fontanka+extra_fontanka:
                    duble.append(j)
            for k in sorted(duble, reverse=True):
                del news_fontanka_extra[k]
            # df1 = pd.concat([df1, pd.DataFrame(news_fontanka_extra)], ignore_index=True)
    except: pass
    df1=pd.DataFrame(news_fontanka+news_fontanka_extra+extra_fontanka)
    print(pd.DataFrame(news_fontanka))
    print(news_fontanka_extra)
    # print(df1)

    
    timestamp = datetime.now().strftime('%Y-%m-%d')
    # df1.to_csv(f"{name}_news_fontanka.csv", index=False, encoding="utf-8-sig")
    df1.to_csv(f"{name}_{inn}_fontanka_{timestamp}.csv", index=False, encoding="utf-8-sig")

    news_dp=[]
    news_dp = parse_dp(driver, name, date_from, date_to)
    df2 = pd.DataFrame(news_dp)
    print(df2)

    extra_dp=[]
    for i in extra_name:
        extra=parse_dp(driver, name, date_from, date_to, i)
        extra_dp=extra_dp+extra
    duble=[]
    for j in range(len(extra_dp)):
        if extra_dp[j] in news_dp:
            duble.append(j)
    for k in sorted(duble, reverse=True):
        del extra_dp[k]

    news_dp_extra=[]
    # df2.to_csv(f"{name}_news_dp.csv", index=False, encoding="utf-8-sig")
    
    try:
        if not(pd.isna(names.iloc[i, 1])):
            news_dp_extra = parse_dp(driver, name, date_from, date_to, names.iloc[i, 1])
            duble=[]
            for j in range(len(news_dp_extra)):
                if news_dp_extra[j] in news_dp+extra_dp:
                    duble.append(j)
            for k in sorted(duble, reverse=True):
                del news_dp_extra[k]
            # df2 = pd.concat([df2, pd.DataFrame(news_dp_extra)],ignore_index=True)
    except: pass
    df2=pd.DataFrame(news_dp+news_dp_extra+extra_dp)
    print(df2)
    # df2.to_csv(f"{name}_news_dp.csv", index=False, encoding="utf-8-sig")
    df2.to_csv(f"{name}_{inn}_dp_{timestamp}.csv", index=False, encoding="utf-8-sig")


    all_news = news_fontanka + news_dp+extra_dp+extra_fontanka+news_dp_extra+news_fontanka_extra

    df = pd.DataFrame(all_news)
    df.to_csv(f"{name}_news.csv", index=False, encoding="utf-8-sig")
    
    print(f"Найдено {len(all_news)} новостей:")
    for news in all_news:
        print(f"Компания: {news['CompanyName']}")
        print(f"Источник: {news['NewsSource']}")
        print(f"Заголовок: {news['NewsTitle']}")
        print(f"Дата: {news['NewsDate']}")
        print(f"Ссылка: {news['NewsURL']}")
        print("---")
driver.quit()

                                CompanyName         INN  \
7            Теплосеть Санкт-Петербурга, АО  7810577007   
8         Единая теплосетевая компания, ООО  7804692788   
9             Газпром газораспределение, АО  7838306818   
10                          Ресурс АТЭ, ООО  7810900796   
11  Региональная теплосетевая компания, ООО  7811739941   
12                            Эко лэнд, ООО  7839031482   
13                Петербургтеплоэнерго, ООО  7838024362   
14                     Леноблводоканал, ГУП  4703144282   
15                        Петербурггаз, ООО  7838017541   
18                          Промотходы, ЗАО  4703061004   
19                                ЛОТЭК, АО  4716028445   
22                   Гарант-Сестрорецк, ООО  7813588293   

                                      URL  Region             RegionName  \
7             https://www.teplosetspb.ru/      78     г. Санкт-Петербург   
8                    https://etkteplo.ru/      78     г. Санкт-Петербург   
9   

In [15]:
# Исправление кодировки
import os
import csv
import chardet

def process_folder(folder_path, column_name):
    for file in os.listdir(folder_path):
        if not file.lower().endswith('.csv'):
            continue
        csv_path = os.path.join(folder_path, file)
        print(f"\n📂 Проверяю файл: {file}")
        try:
            df=pd.read_csv(csv_path)
            # df.dropna(how='all', inplace=True)
            for i in range(len(df)):
                try:
                    df.iloc[i, 5]=df.iloc[i, 5].encode("mac_roman").decode("utf-8", errors="replace")
                except: pass
            
            print(df)
            df.to_csv(csv_path)
        except: pass
if __name__ == "__main__":
    industry="Комунальные услуги"
    folder_path = industry      # 📁 укажи путь к папке
    column_name = "text"       # 🔤 название столбца для проверки
    process_folder(folder_path, column_name)



📂 Проверяю файл: Невский экологический оператор, АО_news.csv
   NewsFinder                         CompanyName  \
0      парсер  Невский экологический оператор, АО   
1      парсер  Невский экологический оператор, АО   
2      парсер  Невский экологический оператор, АО   
3      парсер  Невский экологический оператор, АО   
4      парсер  Невский экологический оператор, АО   
..        ...                                 ...   
66     парсер  Невский экологический оператор, АО   
67     парсер  Невский экологический оператор, АО   
68     парсер  Невский экологический оператор, АО   
69     парсер  Невский экологический оператор, АО   
70     парсер  Невский экологический оператор, АО   

                                            NewsTitle                NewsDate  \
0   К 2030 году Петербург выйдет на плато по доле ...  00:17, 01 декабря 2025   
1   Смольный вслед за "Сбером" выкупил "зелёные" о...   13:49, 27 ноября 2025   
2   "Сбер" выкупил выпуск "зелёных" облигаций под ...   17